# Netflix Data Analytics Project

This notebook processes and analyzes Netflix's dataset to prepare it for data analytics. It handles:
- Data loading and exploration
- Missing value imputation
- Feature engineering 
- Data cleaning and standardization
- Export to filtered dataset for analysis

In [57]:
import pandas as pd

## Step 1: Import Libraries

Import pandas for data manipulation and analysis.

In [58]:
df=pd.read_csv("netflix.csv")
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


## Step 2: Load and Explore Data

Load the Netflix dataset from CSV file and display first few rows to understand the structure.

In [59]:
df.isnull().sum()

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

## Step 3: Check for Missing Values

Analyze missing values in each column to understand data quality issues.

In [60]:
df.dtypes

show_id         object
type            object
title           object
director        object
cast            object
country         object
date_added      object
release_year     int64
rating          object
duration        object
listed_in       object
description     object
dtype: object

## Step 4: Check Data Types

Verify the data types of each column to ensure proper treatment during analysis.

In [61]:
df["director"].fillna("Unknown", inplace=True)
df["cast"].fillna("Unknown", inplace=True)
df["country"].fillna("Unknown", inplace=True)
df["rating"].fillna(df["rating"].mode()[0], inplace=True)
df["listed_in"].fillna("Unknown", inplace=True)

/var/folders/fy/sj43z68s65g67n7z6vwkdtrw0000gn/T/ipykernel_12423/3050146860.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["director"].fillna("Unknown", inplace=True)
/var/folders/fy/sj43z68s65g67n7z6vwkdtrw0000gn/T/ipykernel_12423/3050146860.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always

## Step 5: Handle Missing Values - Phase 1

Fill missing values in categorical columns with appropriate defaults:
- **director**: Replace NaN with "Unknown"
- **cast**: Replace NaN with "Unknown"
- **country**: Replace NaN with "Unknown"
- **rating**: Fill with the most common rating value (mode)
- **listed_in**: Replace NaN with "Unknown"

In [62]:
df["country"] = df["country"].str.split(", ")
df = df.explode("country")
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


## Step 6: Expand Country Column

Netflix content can be from multiple countries stored as comma-separated values. Split and explode them into separate rows to enable country-level analysis.

In [63]:
# Duration split
df["duration_num"] = df["duration"].str.extract(r'(\d+)').astype(float)
df["duration_type"] = df["duration"].str.extract(r'([a-zA-Z]+)')

# Year added
df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce")
df["year_added"] = df["date_added"].dt.year

# Trim spaces
df["country"] = df["country"].str.strip()
df["listed_in"] = df["listed_in"].str.strip()

## Step 7: Feature Engineering

Extract and transform key features from existing columns:
- **duration_num**: Extract numeric value from duration string (e.g., "90" from "90 min")
- **duration_type**: Extract duration type (min/Season)
- **year_added**: Extract year when content was added to Netflix from date_added
- **Text Cleaning**: Remove leading/trailing whitespace from country and listed_in

In [64]:
df.isnull().sum()

show_id            0
type               0
title              0
director           0
cast               0
country            0
date_added       111
release_year       0
rating             0
duration           3
listed_in          0
description        0
duration_num       3
duration_type      3
year_added       111
dtype: int64

## Step 8: Verify Missing Values After Processing

Check if any null values remain after the initial data cleaning phase.

In [65]:
# Fix date
df["year_added"] = df["date_added"].dt.year
df["date_added"] = df["date_added"].astype(str).str.strip()
df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce")

df["year_added"] = df["date_added"].dt.year
df["year_added"].fillna(0, inplace=True)

# Fix duration
df["duration"].fillna("0 min", inplace=True)
df["duration_num"] = df["duration"].str.extract(r'(\d+)').astype(float)

/var/folders/fy/sj43z68s65g67n7z6vwkdtrw0000gn/T/ipykernel_12423/2051273963.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["year_added"].fillna(0, inplace=True)
/var/folders/fy/sj43z68s65g67n7z6vwkdtrw0000gn/T/ipykernel_12423/2051273963.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always beha

## Step 9: Handle Remaining Missing Values - Phase 2

Fix remaining missing values that weren't caught in phase 1:
- **year_added**: Convert to integer year format and fill NaN with 0
- **duration**: Fill remaining NaN values with "0 min"
- **duration_num**: Extract numeric values for all durations again to catch any missed values

In [66]:
df.isnull().sum()

show_id            0
type               0
title              0
director           0
cast               0
country            0
date_added       111
release_year       0
rating             0
duration           0
listed_in          0
description        0
duration_num       0
duration_type      3
year_added         0
dtype: int64

## Step 10: Final Null Check After Phase 2

Verify that missing values have been properly handled after the second phase of cleaning.

In [67]:
df["duration_type"].fillna("Unknown", inplace=True)

/var/folders/fy/sj43z68s65g67n7z6vwkdtrw0000gn/T/ipykernel_12423/1897196219.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["duration_type"].fillna("Unknown", inplace=True)


## Step 11: Handle Missing Duration Type

Fill any remaining missing `duration_type` values (Minutes vs Seasons) with "Unknown".

In [68]:
df["date_added"] = df["date_added"].fillna(pd.Timestamp("1900-01-01"))

## Step 12: Handle Missing Date Added

Replace any remaining NaN values in `date_added` with a default historical date (1900-01-01) to represent content added before proper record keeping.

In [70]:
df.isnull().sum()

show_id          0
type             0
title            0
director         0
cast             0
country          0
date_added       0
release_year     0
rating           0
duration         0
listed_in        0
description      0
duration_num     0
duration_type    0
year_added       0
dtype: int64

## Step 13: Final Verification

Confirm that all missing values have been successfully handled and the dataset is ready for analysis.

In [71]:
df.to_csv("netflix_filtered.csv",index=False)

## Step 14: Export Cleaned Data

Save the fully processed and cleaned dataset to a new CSV file (`netflix_filtered.csv`) for downstream analysis and visualizations.